# Crepúsculos Astronómicos
Cálculo de hora de orto, ocaso y crepúsculos náuticos para toma de alturas estelares.

Este simulador está diseñado para fines educativos. **No lo utilices para la navegación real.**

<a href="https://colab.research.google.com/github/jorgejuan007/Nautica/blob/main/simulaciones/70_calculo_crepusculos_astronomicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
from datetime import datetime

def calcular_orto_ocaso(dia_juliano, lat_deg, lon_deg):
    # Algoritmo simplificado para estimación educativa.
    # No usar para navegación real (no incluye correcciones por refracción complejas).
    
    # Declinación aproximada del sol
    dias_desde_equinoccio = dia_juliano - 80
    declinacion = 23.45 * np.sin(np.radians(360/365 * dias_desde_equinoccio))
    dec_rad = np.radians(declinacion)
    lat_rad = np.radians(lat_deg)
    
    # Hora de paso por el meridiano
    # Ecuación del tiempo muy simplificada
    b = np.radians(360/365 * (dia_juliano - 81))
    eot = 9.87 * np.sin(2*b) - 7.53 * np.cos(b) - 1.5 * np.sin(b)
    
    # Ángulo horario para altura h
    def hora_para_altura(h_deg):
        h_rad = np.radians(h_deg)
        cos_ha = (np.sin(h_rad) - np.sin(lat_rad)*np.sin(dec_rad)) / (np.cos(lat_rad)*np.cos(dec_rad))
        if cos_ha > 1 or cos_ha < -1:
            return None # El sol nunca alcanza esta altura (noches blancas o días oscuros)
        return np.degrees(np.arccos(cos_ha)) / 15 # en horas
        
    # Alturas estándar
    # Orto/Ocaso: -0.833 grados (por refracción y semidiámetro)
    # Crepúsculo Civil: -6 grados
    # Crepúsculo Náutico: -12 grados (Ideal para medir estrellas con sextante, se ve horizonte y estrellas)
    
    ha_orto = hora_para_altura(-0.833)
    ha_civil = hora_para_altura(-6)
    ha_nautico = hora_para_altura(-12)
    
    if ha_orto is None:
        print("El sol no sale ni se pone en esta latitud hoy (Sol de medianoche o noche polar).")
        return
        
    # UTC del mediodía local
    noon_utc = 12 - (lon_deg / 15) - (eot / 60)
    
    print(f"Datos para Latitud {lat_deg}º, Longitud {lon_deg}º (Día {dia_juliano}/365)")
    print("--- Todas las horas en formato decimal UTC ---")
    
    print(f"Paso por el meridiano (Mediodía UTC): {noon_utc:.2f}")
    
    if ha_nautico:
        print(f"Inicio Crepúsculo Náutico matutino: {noon_utc - ha_nautico:.2f} UTC (Horizonte visible, medir estrellas)")
    if ha_civil:
        print(f"Inicio Crepúsculo Civil matutino: {noon_utc - ha_civil:.2f} UTC")
    
    print(f"Orto (Amanecer): {noon_utc - ha_orto:.2f} UTC")
    print(f"Ocaso (Atardecer): {noon_utc + ha_orto:.2f} UTC")
    
    if ha_civil:
        print(f"Fin Crepúsculo Civil vespertino: {noon_utc + ha_civil:.2f} UTC")
    if ha_nautico:
        print(f"Fin Crepúsculo Náutico vespertino: {noon_utc + ha_nautico:.2f} UTC (Último momento para sextante)")

dia = widgets.IntSlider(value=180, min=1, max=365, description='Día Año:')
lat = widgets.FloatSlider(value=36.0, min=-89, max=89, description='Latitud (º):')
lon = widgets.FloatSlider(value=-6.0, min=-180, max=180, description='Longitud (º):')

out = widgets.interactive_output(calcular_orto_ocaso, {'dia_juliano': dia, 'lat_deg': lat, 'lon_deg': lon})
display(widgets.VBox([dia, lat, lon, out]))
